In [2]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

path_to_dumps = os.environ['DATAPATH']

this notebook aims to see the state of the wikipedia text dumps and apply any required cleaning of the text before tokenisation.

In [3]:
dumps_filenames = os.listdir(path_to_dumps)
dump_example = pd.read_parquet(f'{path_to_dumps}/train-00000-of-00041.parquet')

In [12]:
dump_example

,id,url,title,text
0,12,https://en.wikipedia.org/wiki/Anarchism,Anarchism,Anarchism is a political philosophy and moveme...
1,39,https://en.wikipedia.org/wiki/Albedo,Albedo,Albedo (; ) is the fraction of sunlight that i...
2,290,https://en.wikipedia.org/wiki/A,A,"A, or a, is the first letter and the first vow..."
3,303,https://en.wikipedia.org/wiki/Alabama,Alabama,Alabama () is a state in the Southeastern regi...
4,305,https://en.wikipedia.org/wiki/Achilles,Achilles,"In Greek mythology, Achilles ( ) or Achilleus ..."
...,...,...,...,...
156284,5275580,https://en.wikipedia.org/wiki/Brookins%20Campbell,Brookins Campbell,"Brookins Campbell (1808December 25, 1853) was ..."
156285,5275584,https://en.wikipedia.org/wiki/Antonio%20Valencia,Antonio Valencia,"Luis Antonio Valencia Mosquera, known as Anton..."
156286,5275590,https://en.wikipedia.org/wiki/Geraint%20Bowen,Geraint Bowen,Geraint Bowen may refer to:\n\nGeraint Bowen (...
156287,5275592,https://en.wikipedia.org/wiki/Trstenik%20Airport,Trstenik Airport,"Trstenik Airport (, Latin: Aerodrom Trstenik)..."


so the data has the URL, the title and the text. my initial reaction is that the title and url are probably largely irrelevant. there is a few different considerations we'll have to take regarding the text. my main thoughts are
1. information on articles should be relatively interconnected (that's why things like wiki races are a thing!)
2. so the network should be able to gain general information about, ships and guns, for example, and then possibly infer information about warships (given warships is held out as a validation set)
3. the network should be able to learn the structure of english fairly easily using wikipedia.

for number 3 to be true, we might need to do some cleaning of the existing text, as some of the above examples contain some artifacts.

In [15]:
print(dump_example.iloc[1]['text'][0:100])

Albedo (; ) is the fraction of sunlight that is diffusely reflected by a body. It is measured on a s


so these artifacts come from the pronunciation stuff thats at the start of the page. claude can write me a fairly good cleaning function using some fuzzy matching to get rid of most of the artifacts that exist. some other examples are below.

In [19]:
print(dump_example.iloc[1]['text'][-1000:-1])

ry surface to a depth of a meter or so (a few wavelengths of the radar wavelength which is typically at the decimeter scale) using the following empirical relationships:

.

History 
The term albedo was introduced into optics by Johann Heinrich Lambert in his 1760 work Photometria.

See also

 Cool roof
 Daisyworld
 Emissivity
 Exitance
 Global dimming
 Ice–albedo feedback
 Irradiance
 Kirchhoff's law of thermal radiation
 Opposition surge
 Polar see-saw
 Radar astronomy
 Solar radiation management

References

External links

 Albedo Project 
 Albedo – Encyclopedia of Earth
 NASA MODIS BRDF/albedo product site
 Ocean surface albedo look-up-table
 Surface albedo derived from Meteosat observations
 A discussion of Lunar albedos
 reflectivity of metals (chart) 

Land surface effects on climate
Climate change feedbacks
Climate forcing
Climatology
Electromagnetic radiation
Meteorological quantities
Radiometry
Scattering, absorption and radiative transfer (optics)
Radiation
1760s neologism


at the bottom of wikipedia pages there are 'see also' sections which need to be filtered. the below function will be added in `src/askwiki/data/utils/clean.py`

In [11]:
import re

TAIL_SECTIONS = re.compile(
    r'\n\s*(See also|References|External links|Further reading|Notes|Footnotes|'
    r'Citations|Sources|Bibliography|Notes and references)\s*\n.*', re.S)
EMPTY_PARENS = re.compile(r'\(\s*(?:[;,]\s*)*\)')
LEAD_PUNCT_PAREN = re.compile(r'\(\s*(?:[;,]\s*)+')
TRAIL_PUNCT_PAREN = re.compile(r'(?:\s*[;,])+\s*\)')
LINE_BREAKS = re.compile(r'\r\n?|[\u2028\u2029\x0b\x0c\x85]')
ODD_SPACES = re.compile(r'[\xa0\u2000-\u200a\u202f\u205f\u3000]')
ZERO_WIDTH = re.compile(r'[\u200b\u200c\u200d\ufeff]')
MONTHS = 'January|February|March|April|May|June|July|August|September|October|November|December'
GLUED_DATE = re.compile(rf'(\d)({MONTHS})')
TEMPLATES = re.compile(r'\{\{[^{}]*\}\}')
HTML = re.compile(r'<ref[^>]*/>|<ref[^>]*>.*?</ref>|<[^>]+>', re.S)
TABLE_ROWS = re.compile(r'^\s*(\{\||\|\}|\|-|\|).*$', re.M)
MULTISPACE = re.compile(r'[ \t]{2,}')
SPACE_PUNCT = re.compile(r' +([,.;:])')
BLANK_LINES = re.compile(r'\n{3,}')


def clean_text(text: str) -> str:
    text = LINE_BREAKS.sub('\n', text)
    text = ODD_SPACES.sub(' ', text)
    text = ZERO_WIDTH.sub('', text)
    text = TAIL_SECTIONS.sub('', text)
    text = HTML.sub('', text)
    for _ in range(3):
        text = TEMPLATES.sub('', text)
    text = TABLE_ROWS.sub('', text)
    text = text.replace("'''", '').replace("''", '')
    text = LEAD_PUNCT_PAREN.sub('(', text)
    text = TRAIL_PUNCT_PAREN.sub(')', text)
    text = EMPTY_PARENS.sub('', text)
    text = GLUED_DATE.sub(r'\1 – \2', text)
    text = MULTISPACE.sub(' ', text)
    text = SPACE_PUNCT.sub(r'\1', text)
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n[ \t]+', '\n', text)
    text = BLANK_LINES.sub('\n\n', text)
    return text.strip()

let's take a look at some examples of text now.

In [34]:
dump_example['text'] = dump_example['text'].apply(clean_text)

In [35]:
import numpy as np

for idx in np.random.randint(0, len(dump_example), size=15): 
    print(dump_example.iloc[idx]['text'][0:100])

Lawrin (May 15, 1935 – September 1, 1955) was an American thoroughbred racehorse owned by Herbert M.
Individual Learning Plan or ILP is a user (student) specific program or strategy of education or lea
Unna Guovdelisjávri or Vuolip Čoarvejávri is a lake that is located on the border of Norway and Swed
Eugen Napoleon Neureuther (13 January 1806 – 23 March 1882) was a German painter, etcher, and illust
Carlos Bernardo Chasseing (25 September 1926 – 16 April 2018) was de facto Federal Interventor of Có
The left colic artery is a branch of the inferior mesenteric artery distributed to the descending co
The Rheinhöhenweg Trail is a popular hiking trail on the mountains of the
Rhine Valley. It leads fro
The discography of Nirvana, an American rock band, consists of three studio albums, twenty-one singl
Arthur Hewlett (12 March 1907 – 25 February 1997) was a British actor.

Hewlett made his stage debut
The New Zealand Professional Firefighters Union (NZPFU) is a trade union providing professi

looks like the text is pretty well normalised now. new lines are restricted to only one new line max separated... looks like foreign languages are still included but that should be fine given our tokenisation method suits this. we now clean each and every file, to generate one large, clean file dump of wikipedia text ready for tokeniser training.

In [15]:
dumps = []

for fn in dumps_filenames:
    print(f'starting {fn}')
    dump_df = pd.read_parquet(f'{path_to_dumps}/{fn}', columns=['id', 'text'])
    dump_df['text'] = dump_df['text'].apply(clean_text)
    dumps.append(dump_df)

dumps_df: pd.DataFrame = pd.concat(dumps)
dumps_df = dumps_df.reset_index(drop=True)
dumps_df.to_parquet(f'{path_to_dumps}/clean.parquet')

starting train-00000-of-00041.parquet
starting train-00001-of-00041.parquet
starting train-00002-of-00041.parquet
starting train-00003-of-00041.parquet
starting train-00004-of-00041.parquet
starting train-00005-of-00041.parquet
starting train-00006-of-00041.parquet
starting train-00007-of-00041.parquet
starting train-00008-of-00041.parquet
starting train-00009-of-00041.parquet
starting train-00010-of-00041.parquet
starting train-00011-of-00041.parquet
starting train-00012-of-00041.parquet
starting train-00013-of-00041.parquet
starting train-00014-of-00041.parquet
starting train-00015-of-00041.parquet
starting train-00016-of-00041.parquet
starting train-00017-of-00041.parquet
starting train-00018-of-00041.parquet
starting train-00019-of-00041.parquet
starting train-00020-of-00041.parquet
starting train-00021-of-00041.parquet
starting train-00022-of-00041.parquet
starting train-00023-of-00041.parquet
starting train-00024-of-00041.parquet
starting train-00025-of-00041.parquet
starting tra

we can now split the data set into a training, validation and test set. a validation set should be okay with < 10m tokens. we can reason a target number of tokens for the validation set through some assumptions. assume we have 2,048 token sequences. we must come to a consensus on the standard deviation of the mean loss per sequence. if we assume $\bar{\sigma}_{seq} = 0.5$, then for 1,000 sequences we have a standard error of $0.5 / \sqrt{1000} = 0.016$, 5,000 yields $0.007$ and 25,000 yields $0.003$.

with each sequence holding 2,048 tokens, this corresponds to ~2 million, ~10 million and ~50 million tokens. given the standard error, we aim for ~10 million tokens; thus requiring ~10 million words (realistically less, due to subword tokenisation).

In [4]:
import duckdb

src = f'{path_to_dumps}/clean.parquet'
con = duckdb.connect()
con.execute("SET memory_limit='24GB'")

we'd like to sample articles as a validation and test set from the overall data but face RAM constraints (32GB), so we use `duckdb` to get over this.

In [5]:
avg_number_of_words = con.execute(
    rf"""
    select avg(len(regexp_extract_all(text, '\S+')))
    from '{src}'
    """
).fetchone()[0]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
print(f'average number of words: {avg_number_of_words}')

average number of words: 438.6725572870873


so we should sample roughly `10,000,000 / 439 = 22,779` articles; call it 20,000 articles. to get 20,000 articles for validation and testing (each), we can use a hash (which is random) on the ID of the articles, and then order the articles by hash, which should provide a random sort. then, we can split up the first 20,000 rows into validation and the next 20,000 rows into test.

In [ ]:
# get the id and the hashed id from the source data frame, and then get the row number (ordering) of the hashed
# article ids. first 20k is validation, rest is test.
con.execute(
    f"""
    create or replace temp table holdout 
    as select id, case when row_number() over (order by h, id) <= 20000 then 'val' else 'test' end as split
    from (select id, hash(id) as h from '{src}' order by h, id limit 40000)
    """
)

# get the validation and test splits
for name in ['val', 'test']:
    con.execute(
        f"""
        copy (select s.* from '{src}' s join holdout h using (id) where h.split = '{name}') to '{path_to_dumps}/{name}.parquet' (format parquet)
        """
    )

# remaining rows are for training; any id that is not in holdout (antijoin) is a training row
con.execute(
    f"""
    copy (select s.* from '{src}' s anti join holdout h using (id)) to '{path_to_dumps}/train.parquet' (format parquet)
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))